<a href="https://colab.research.google.com/github/redouanelg/climatsuds-tutorials/blob/main/Data_Access/imerg_v7_daily_west_africa_access.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Accessing daily IMERG v7 precipitation over West Africa
# Accéder aux précipitations quotidiennes IMERG v7 sur l'Afrique de l'Ouest

**English.** [GPM IMERG](https://gpm.nasa.gov/data/imerg) is NASA's global,
half-hourly to daily precipitation product. This notebook opens the **Final
daily** product, **version 7** (`GPM_3IMERGDF.07`, 0.1° ≈ 10 km), straight from
the cloud and maps it over **West Africa**, where the rainfall comes almost
entirely from the summer monsoon. The data lives in the NASA GES DISC archive
and is mirrored on the [AWS Open Data registry](https://registry.opendata.aws/nasa-gpm3imergdf/)
(bucket `gesdisc-cumulus-prod-protected`, region `us-west-2`).

The dataset is **not** anonymous: you need a **free NASA Earthdata Login**.
We use the [`earthaccess`](https://earthaccess.readthedocs.io) library, which
handles the login and reads the files directly from the cloud — via S3 when you
run inside AWS `us-west-2`, or over HTTPS from anywhere else (Colab, your
laptop). No manual download required.

**Français.** [GPM IMERG](https://gpm.nasa.gov/data/imerg) est le produit de
précipitations mondial de la NASA, de la demi-heure au jour. Ce notebook ouvre
le produit **Final quotidien**, **version 7** (`GPM_3IMERGDF.07`, 0,1° ≈ 10 km),
directement depuis le cloud et le cartographie sur l'**Afrique de l'Ouest**, où
les pluies proviennent presque uniquement de la mousson d'été. Les données sont
archivées au NASA GES DISC et répliquées sur le
[registre AWS Open Data](https://registry.opendata.aws/nasa-gpm3imergdf/)
(bucket `gesdisc-cumulus-prod-protected`, région `us-west-2`).

Le jeu de données n'est **pas** en accès anonyme : il faut un **compte gratuit
NASA Earthdata Login**. Nous utilisons la librairie
[`earthaccess`](https://earthaccess.readthedocs.io), qui gère l'authentification
et lit les fichiers directement depuis le cloud — via S3 si vous êtes dans AWS
`us-west-2`, ou via HTTPS depuis n'importe où ailleurs (Colab, votre ordinateur).
Aucun téléchargement manuel n'est nécessaire.

In [ ]:
# EN: install the cloud-access + analysis stack.  FR: installer les librairies d'accès cloud et d'analyse.
!pip install -q earthaccess "xarray[complete]" h5netcdf dask cartopy matplotlib

## 1 — Authenticate with NASA Earthdata Login

**English.** Create a free account at
[urs.earthdata.nasa.gov](https://urs.earthdata.nasa.gov), then **authorize the
"NASA GESDISC DATA ARCHIVE" application** in your Earthdata profile
(*Applications → Authorized Apps*) — GPM data is served by GES DISC and this
one-time step is a common stumbling block. `earthaccess.login()` will look for a
`~/.netrc` entry, the `EARTHDATA_USERNAME` / `EARTHDATA_PASSWORD` environment
variables, or simply prompt you (which is what happens on Colab).

**Français.** Créez un compte gratuit sur
[urs.earthdata.nasa.gov](https://urs.earthdata.nasa.gov), puis **autorisez
l'application « NASA GESDISC DATA ARCHIVE »** dans votre profil Earthdata
(*Applications → Authorized Apps*) — les données GPM sont servies par GES DISC
et cette étape unique est souvent oubliée. `earthaccess.login()` cherchera une
entrée `~/.netrc`, les variables d'environnement `EARTHDATA_USERNAME` /
`EARTHDATA_PASSWORD`, ou vous demandera vos identifiants (cas de Colab).

In [ ]:
import earthaccess
import xarray as xr

# EN: prompts for your Earthdata Login the first time.  FR: demande vos identifiants Earthdata la première fois.
auth = earthaccess.login()

## 2 — Search the granules over West Africa

**English.** We query NASA's Common Metadata Repository by `short_name` and
`version`, restricting to **September 2023** (the peak of the West African
monsoon) and to a **West Africa bounding box**. The `bounding_box` tuple is
`(min_lon, min_lat, max_lon, max_lat)` — here longitudes **-20°→20°** and
latitudes **0°→28°**, from the Gulf of Guinea coast up into the Sahara. Each
result is one daily file.

**Français.** Nous interrogeons le *Common Metadata Repository* de la NASA par
`short_name` et `version`, en nous limitant à **septembre 2023** (le pic de la
mousson ouest-africaine) et à une **emprise sur l'Afrique de l'Ouest**. Le tuple
`bounding_box` est `(lon_min, lat_min, lon_max, lat_max)` — ici des longitudes
**-20°→20°** et des latitudes **0°→28°**, de la côte du golfe de Guinée jusqu'au
Sahara. Chaque résultat correspond à un fichier quotidien.

In [ ]:
# EN: West Africa search window.  FR: fenêtre de recherche sur l'Afrique de l'Ouest.
results = earthaccess.search_data(
    short_name="GPM_3IMERGDF",
    version="07",
    temporal=("2023-09-01", "2023-09-30"),
    bounding_box=(-20, 0, 20, 28),  # (min_lon, min_lat, max_lon, max_lat)
)
print(f"{len(results)} daily granules found")
results[0]

**English (optional).** Each granule also exposes its native **S3 URI** in the
AWS Open Data bucket. `earthaccess.open()` below uses these automatically when
you run inside AWS `us-west-2`; from anywhere else it transparently falls back to
HTTPS. The object layout in the bucket is:

```
s3://gesdisc-cumulus-prod-protected/GPM_L3/GPM_3IMERGDF.07/2023/09/
    3B-DAY.MS.MRG.3IMERG.20230901-S000000-E235959.V07B.nc4
```

**Français (optionnel).** Chaque granule expose aussi son **URI S3** natif dans
le bucket AWS Open Data. `earthaccess.open()` ci-dessous les utilise
automatiquement si vous êtes dans AWS `us-west-2` ; ailleurs, il bascule de façon
transparente vers HTTPS. L'arborescence des objets dans le bucket est :

```
s3://gesdisc-cumulus-prod-protected/GPM_L3/GPM_3IMERGDF.07/2023/09/
    3B-DAY.MS.MRG.3IMERG.20230901-S000000-E235959.V07B.nc4
```

In [ ]:
# EN: peek at the direct S3 URI of the first granule.  FR: voir l'URI S3 direct du premier granule.
results[0].data_links(access="direct")

## 3 — Open the granules straight from the cloud

**English.** `earthaccess.open()` returns file-like objects we hand to
`xarray.open_mfdataset`, which lazily stacks the 30 daily files along `time`.
We read with the `h5netcdf` engine (the files are NetCDF-4/HDF5) and sort by
time to be safe.

> ⚠️ **Gotcha:** IMERG stores its grids as **`(time, lon, lat)`** — longitude
> *before* latitude. We therefore always select and plot with **named**
> dimensions (`lat=`, `lon=`, `x="lon"`, `y="lat"`) so the orientation is never
> ambiguous.

**Français.** `earthaccess.open()` renvoie des objets de type fichier que nous
passons à `xarray.open_mfdataset`, qui empile paresseusement les 30 fichiers
quotidiens le long de `time`. Nous lisons avec le moteur `h5netcdf` (les fichiers
sont en NetCDF-4/HDF5) et trions par temps par précaution.

> ⚠️ **Piège :** IMERG stocke ses grilles en **`(time, lon, lat)`** — la
> longitude *avant* la latitude. Nous sélectionnons et traçons donc toujours
> avec des dimensions **nommées** (`lat=`, `lon=`, `x="lon"`, `y="lat"`) pour
> lever toute ambiguïté d'orientation.

In [ ]:
files = earthaccess.open(results)

ds = xr.open_mfdataset(
    files,
    engine="h5netcdf",
    combine="nested",
    concat_dim="time",
).sortby("time")

ds

## 4 — Subset to West Africa and pick the precipitation field

**English.** The Final daily product carries several fields; the headline one is
`precipitation` — the daily-mean rate in **mm/day**. We slice the same West
Africa window we searched on. (IMERG latitudes run south→north and longitudes
west→east, so ascending `slice()`s work directly.)

**Français.** Le produit Final quotidien contient plusieurs champs ; le principal
est `precipitation` — le taux quotidien moyen en **mm/jour**. Nous découpons la
même fenêtre ouest-africaine que pour la recherche. (Les latitudes IMERG vont du
sud au nord et les longitudes d'ouest en est, donc des `slice()` croissants
fonctionnent directement.)

In [ ]:
wa = ds.sel(lon=slice(-20, 20), lat=slice(0, 28))
precip = wa["precipitation"]  # dims (time, lon, lat), units mm/day
precip

## 5 — Map the monthly-mean precipitation

**English.** Averaging over the 30 days gives the September mean rainfall field.
The map should show the classic West African monsoon signature: a wet band along
the Guinea coast and Sahel that fades sharply northward into the dry Sahara.

**Français.** En moyennant sur les 30 jours, on obtient le champ de pluie moyen
de septembre. La carte doit montrer la signature classique de la mousson
ouest-africaine : une bande humide le long de la côte guinéenne et du Sahel qui
s'estompe brusquement vers le nord, dans le Sahara aride.

In [ ]:
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature

monthly_mean = precip.mean("time").transpose("lat", "lon").compute()

fig = plt.figure(figsize=(11, 7))
ax = plt.axes(projection=ccrs.PlateCarree())
mesh = monthly_mean.plot.pcolormesh(
    ax=ax,
    x="lon",
    y="lat",
    transform=ccrs.PlateCarree(),
    cmap="YlGnBu",
    vmin=0,
    robust=True,
    cbar_kwargs={
        "label": f"Precipitation ({precip.attrs.get('units', 'mm/day')})",
        "shrink": 0.7,
        "pad": 0.02,
    },
)
ax.add_feature(cfeature.COASTLINE, linewidth=0.6)
ax.add_feature(cfeature.BORDERS, linewidth=0.4)
ax.set_extent([-20, 20, 0, 28], crs=ccrs.PlateCarree())
gl = ax.gridlines(draw_labels=True, linewidth=0.3, color="gray", alpha=0.4)
gl.top_labels = gl.right_labels = False
ax.set_title("IMERG v7 — mean daily precipitation, West Africa, September 2023")
plt.show()

## 6 — Domain-averaged daily time series

**English.** Finally, a single curve: the domain-averaged daily rainfall through
the month. We weight each grid cell by `cos(latitude)` so cells near the top of
the box don't count more than they should.

**Français.** Enfin, une seule courbe : la pluie quotidienne moyennée sur le
domaine au fil du mois. Nous pondérons chaque maille par `cos(latitude)` pour que
les mailles proches du haut de la boîte ne pèsent pas plus que de raison.

In [ ]:
import numpy as np

weights = np.cos(np.deg2rad(wa["lat"]))
weights.name = "weights"

ts = precip.weighted(weights).mean(["lon", "lat"]).compute()

plt.figure(figsize=(10, 4))
ts.plot(marker="o")
plt.title("Area-averaged daily precipitation over West Africa (IMERG v7, Sept 2023)")
plt.ylabel(f"Precipitation ({precip.attrs.get('units', 'mm/day')})")
plt.xlabel("Date")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Going further · Pour aller plus loin

**English**
- Tighten the box to a single country, or swap `temporal=` for another month or a
  multi-year span to build a climatology.
- Other useful fields in the file: `precipitationQualityIndex`,
  `randomError`, `probabilityLiquidPrecipitation`.
- For long time series, prefer the monthly product `GPM_3IMERGM` (lighter), or
  run this notebook on an AWS `us-west-2` instance for direct-S3 speed.

**Français**
- Resserrez l'emprise sur un seul pays, ou changez `temporal=` pour un autre mois
  ou une période pluriannuelle afin de construire une climatologie.
- Autres champs utiles du fichier : `precipitationQualityIndex`, `randomError`,
  `probabilityLiquidPrecipitation`.
- Pour de longues séries, préférez le produit mensuel `GPM_3IMERGM` (plus léger),
  ou exécutez ce notebook sur une instance AWS `us-west-2` pour la vitesse du S3
  direct.

**References / Références**
- AWS Open Data: https://registry.opendata.aws/nasa-gpm3imergdf/
- Dataset DOI: https://doi.org/10.5067/GPM/IMERGDF/DAY/07
- GES DISC dataset page: https://disc.gsfc.nasa.gov/datasets/GPM_3IMERGDF_07/summary
- earthaccess: https://earthaccess.readthedocs.io
- IMERG overview: https://gpm.nasa.gov/data/imerg